In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Nikhil3654/generalizable-llm-planning.git"
BRANCH = "feature/benchmark-index"

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "generalizable-llm-planning"
KAGGLE_INPUT = Path("/kaggle/input")


In [2]:
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Repository ready:", REPO_DIR)


Cloning into '/kaggle/working/generalizable-llm-planning'...


Repository ready: /kaggle/working/generalizable-llm-planning


In [3]:
matches = list(
    KAGGLE_INPUT.rglob("ipc-2000/domains/blocks-strips-untyped/domain.pddl")
)

if not matches:
    raise FileNotFoundError(
        "Could not locate the IPC dataset under /kaggle/input. "
        "Attach the IPC PDDL dataset using Add Input."
    )

if len(matches) > 1:
    print("Multiple IPC dataset matches found:")
    for match in matches:
        print(" -", match)
    print("\nUsing the first match.")

reference_domain = matches[0]

# Expected structure:
# DATASET_ROOT/ipc-2000/domains/blocks-strips-untyped/domain.pddl
DATASET_ROOT = reference_domain.parents[3]

print("Dataset root:", DATASET_ROOT)
print("Reference domain:", reference_domain)


Dataset root: /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances
Reference domain: /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-2000/domains/blocks-strips-untyped/domain.pddl


In [4]:
from src.benchmark import discover_domains

domains = discover_domains(DATASET_ROOT)

print("Discovered domain variants:", len(domains))
print()

for domain in domains[:20]:
    print(
        domain["name"],
        "| instances:",
        domain["num_instances"],
        "|",
        domain["domain_file"],
    )


Discovered domain variants: 283

assembly-round-1-adl | instances: 30 | /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-1998/domains/assembly-round-1-adl/domain.pddl
grid-round-2-strips | instances: 5 | /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-1998/domains/grid-round-2-strips/domain.pddl
gripper-round-1-adl | instances: 20 | /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-1998/domains/gripper-round-1-adl/domain.pddl
gripper-round-1-strips | instances: 20 | /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-1998/domains/gripper-round-1-strips/domain.pddl
logistics-round-1-adl | instances: 30 | /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-1998/domains/logistics-round-1-adl/domain.pddl
logistics-round-1-strips | instances: 35 | /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-1998/domains/logistics-round-1-strips/domain.pddl
logistics-round-2-strips | instanc

In [5]:
import pandas as pd

domain_df = pd.DataFrame([
    {
        "domain_variant": item["name"],
        "num_instances": item["num_instances"],
        "domain_file": str(item["domain_file"].relative_to(DATASET_ROOT)),
    }
    for item in domains
])

print("Rows:", len(domain_df))
display(domain_df.head(20))


Rows: 283


,domain_variant,num_instances,domain_file
0,assembly-round-1-adl,30,ipc-1998/domains/assembly-round-1-adl/domain.pddl
1,grid-round-2-strips,5,ipc-1998/domains/grid-round-2-strips/domain.pddl
2,gripper-round-1-adl,20,ipc-1998/domains/gripper-round-1-adl/domain.pddl
3,gripper-round-1-strips,20,ipc-1998/domains/gripper-round-1-strips/domain...
4,logistics-round-1-adl,30,ipc-1998/domains/logistics-round-1-adl/domain....
5,logistics-round-1-strips,35,ipc-1998/domains/logistics-round-1-strips/doma...
6,logistics-round-2-strips,5,ipc-1998/domains/logistics-round-2-strips/doma...
7,movie-round-1-adl,30,ipc-1998/domains/movie-round-1-adl/domain.pddl
8,movie-round-1-strips,30,ipc-1998/domains/movie-round-1-strips/domain.pddl
9,mystery-prime-round-1-adl,30,ipc-1998/domains/mystery-prime-round-1-adl/dom...


In [6]:
def competition_from_path(relative_path):
    for part in Path(relative_path).parts:
        if part.startswith("ipc-"):
            return part
    return "unknown"

domain_df["competition"] = domain_df["domain_file"].apply(
    competition_from_path
)

competition_summary = (
    domain_df
    .groupby("competition", as_index=False)
    .agg(
        domain_variants=("domain_variant", "count"),
        total_instances=("num_instances", "sum"),
    )
    .sort_values("competition")
)

display(competition_summary)


,competition,domain_variants,total_instances
0,ipc-1998,14,335
1,ipc-2000,12,1392
2,ipc-2002,48,951
3,ipc-2004,30,1350
4,ipc-2006,38,1008
5,ipc-2008,33,990
6,ipc-2011,46,920
7,ipc-2014,62,1216


In [7]:
largest_domains = (
    domain_df
    .sort_values("num_instances", ascending=False)
    .head(20)
)

display(largest_domains)


,domain_variant,num_instances,domain_file,competition
25,schedule-adl-untyped,150,ipc-2000/domains/schedule-adl-untyped/domain.pddl,ipc-2000
18,elevator-strips-simple-typed,150,ipc-2000/domains/elevator-strips-simple-typed/...,ipc-2000
19,elevator-strips-simple-untyped,150,ipc-2000/domains/elevator-strips-simple-untype...,ipc-2000
24,schedule-adl-typed,150,ipc-2000/domains/schedule-adl-typed/domain.pddl,ipc-2000
16,elevator-adl-full-typed,150,ipc-2000/domains/elevator-adl-full-typed/domai...,ipc-2000
17,elevator-adl-simple-typed,150,ipc-2000/domains/elevator-adl-simple-typed/dom...,ipc-2000
15,blocks-strips-untyped,102,ipc-2000/domains/blocks-strips-untyped/domain....,ipc-2000
14,blocks-strips-typed,102,ipc-2000/domains/blocks-strips-typed/domain.pddl,ipc-2000
23,logistics-strips-untyped,84,ipc-2000/domains/logistics-strips-untyped/doma...,ipc-2000
22,logistics-strips-typed,84,ipc-2000/domains/logistics-strips-typed/domain...,ipc-2000


In [8]:
from src.benchmark import build_benchmark_index

benchmark_df = build_benchmark_index(DATASET_ROOT)

print("Total benchmark problems:", len(benchmark_df))
print("Unique domain variants:", benchmark_df["domain_variant"].nunique())
print("Competitions:", benchmark_df["competition"].nunique())

display(benchmark_df.head(20))


ImportError: cannot import name 'build_benchmark_index' from 'src.benchmark' (/kaggle/working/generalizable-llm-planning/src/benchmark.py)

In [ ]:
OUTPUT_PATH = WORKING_DIR / "benchmark_index.csv"

benchmark_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(benchmark_df))


In [ ]:
assert len(benchmark_df) > 0
assert benchmark_df["problem"].notna().all()
assert benchmark_df["domain_file"].notna().all()
assert benchmark_df["problem_file"].notna().all()

duplicates = benchmark_df.duplicated(
    subset=["competition", "domain_variant", "problem"]
).sum()

print("Duplicate benchmark rows:", duplicates)

display(
    benchmark_df.groupby(
        ["competition", "domain_variant"],
        as_index=False
    ).size().head(30)
)
